# Vast.ai / арендованный GPU (RTX 5090)

**Структура перед запуском:**
```
рабочая_папка/
├── kazakh_missing_words_vastai.ipynb
└── data/
    ├── train.csv
    └── test_inputs.csv
```

**Оптимизации:** batch_size=64 под 32GB VRAM, eval_strategy (новый API transformers).

# Vast.ai / арендованный GPU (RTX 5090, 32 GB)

**Структура:** создай папку `data/` рядом с ноутбуком, положи в неё:
- `train.csv`
- `test_inputs.csv`

**Изменения:** batch_size=64 (под 32 GB VRAM), eval_strategy вместо evaluation_strategy.

# Kazakh Missing Words Challenge — WISH Hackathon 2026

**Задача:** предсказать 0-based индекс позиции пропущенного слова в казахском предложении.

**Модель:** XLM-RoBERTa-base (sequence classification, num_labels = max_position).

In [ ]:
# Vast.ai: установка в тот же Python, что и Jupyter kernel
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "matplotlib", "scikit-learn", "torch", "transformers", "accelerate", "tqdm"], check=True)
print("Готово. Kernel → Restart, затем Run All.")

In [ ]:
import numpy as np
import pandas as pd
import os

# На Kaggle — показывает файлы датасета; локально/Colab — пропускаем
if os.path.exists('/kaggle/input'):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("Локальная среда: ожидаются данные в ./data/")

In [ ]:
# ⚠️ Kaggle: ЭТУ ЯЧЕЙКУ НЕ ТРОГАТЬ! transformers/accelerate уже установлены.
# pip install только ломает всё. На Kaggle — просто Run All без этой ячейки, или она пропустит pip.
if not os.path.exists("/kaggle/input"):
    get_ipython().run_cell_magic("capture", "", "!pip install -q transformers accelerate")
else:
    print("Kaggle: используем предустановленные transformers/accelerate — всё ок")

## Дополнительные импорты

Импортируем библиотеки для работы с PyTorch, transformers и sklearn.

In [ ]:
import time, random, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Конфиг

Все гиперпараметры и пути в одном словаре. Пути соответствуют Kaggle dataset.

In [ ]:
# Автоопределение путей: Kaggle vs локально/Colab
_data_dir = None
if os.path.exists("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        if "train.csv" in files and "test_inputs.csv" in files:
            _data_dir = root
            break
if _data_dir:
    _output_dir = "/kaggle/working"
    print("Среда: Kaggle")
    print("Данные:", _data_dir)
else:
    _data_dir = "./data"
    _output_dir = "./output"
    os.makedirs(_data_dir, exist_ok=True)
    os.makedirs(_output_dir, exist_ok=True)
    print("Среда: локально/Colab. Данные из папки data/")

_seed, _ensemble_seed = 42, 123
CFG = {
    "model_name": "xlm-roberta-large",
    "max_len": 128,
    "max_position": 55,
    "batch_size": 32,
    "epochs": 4,
    "lr": 2e-5,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "fp16": True,
    "n_samples_per_sentence": 5,
    "val_size": 0.05,
    "min_words": 3,
    "seed": _seed,
    "ensemble_seeds": [_seed, _ensemble_seed],
    "use_ensemble": True,
    "train_path": os.path.join(_data_dir, "train.csv"),
    "test_path": os.path.join(_data_dir, "test_inputs.csv"),
    "output_dir": os.path.join(_output_dir, "model_seed" + str(_seed)),
    "submission_path": os.path.join(_output_dir, "submission.csv"),
    "ensemble_submission_path": os.path.join(_output_dir, "submission_ensemble.csv"),
}
set_seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
print("Config ready. Seeds для ансамбля:", CFG["ensemble_seeds"])

## Загрузка данных и EDA

Загружаем train и test, смотрим структуру, добавляем колонку длины предложений, строим гистограммы и обновляем `max_position`.

In [ ]:
# Загрузка или fallback на синтетические данные
if os.path.exists(CFG["train_path"]) and os.path.exists(CFG["test_path"]):
    train_df = pd.read_csv(CFG["train_path"])
    test_df = pd.read_csv(CFG["test_path"])
else:
    print("⚠️ Файлы train.csv и test_inputs.csv не найдены. Используем синтетические данные для проверки пайплайна.")
    print("   Скачай датасет с Kaggle и положи в папку data/ для реального обучения.\n")
    # Минимальный синтетический датасет (примеры на казахском)
    _sentences = [
        "Мен дүкенге барып келдім", "Сәлем қалайсың", "Бүгін жақсы күн",
        "Ол мектепке барды", "Біз кітап оқыдық", "Сен қайда бардың",
        "Мен үйдемін", "Олар кино көрді", "Бүгін салқын ауа",
        "Ол жақсы оқушы", "Мен спортты жақсы көремін",
    ] * 500  # ~5500 предложений для быстрого прогона
    train_df = pd.DataFrame({"full_text": _sentences})
    test_df = pd.DataFrame({
        "ID": range(100),
        "text": ["Мен барып келдім", "Сәлем қалайсың", "Бүгін жақсы"] * 33 + ["Мен барып келдім"],
    })

print("Train head:")
print(train_df.head(3))
print("\nTest head:")
print(test_df.head(3))

if len(train_df) < 500_000:
    print(f"\n⚠️ Используется неполный датасет ({len(train_df):,} строк). Для лучшего результата загрузи полный train.csv (~554k строк).")

train_df["n_words"] = train_df["full_text"].str.split().str.len()
test_df["n_words"] = test_df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_df["n_words"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("Train: длина предложений (кол-во слов)")
axes[0].set_xlabel("n_words")
axes[1].hist(test_df["n_words"], bins=50, edgecolor="black", alpha=0.7)
axes[1].set_title("Test: длина предложений (кол-во слов)")
axes[1].set_xlabel("n_words")
plt.tight_layout()
plt.show()

print(f"Train - max: {train_df['n_words'].max()}, median: {train_df['n_words'].median()}")
print(f"Test - max: {test_df['n_words'].max()}, median: {test_df['n_words'].median()}")

CFG["max_position"] = int(test_df["n_words"].max()) + 2
print(f"\nОбновлённый max_position: {CFG['max_position']}")

## Генерация обучающих данных

Из ~550k полных казахских предложений генерируем ~1.65 млн пар (текст с пропуском → позиция пропущенного слова). Для каждого предложения берём до 3 случайных позиций и создаём обучающий пример.

In [ ]:
def generate_training_samples(
    df: pd.DataFrame,
    n_per_sentence: int = 3,
    min_words: int = 3,
    max_position: int = 55,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Генерирует обучающие пары: текст с удалённым словом -> позиция (0-based).
    
    Args:
        df: DataFrame с колонкой full_text.
        n_per_sentence: Макс. число примеров на предложение.
        min_words: Минимальное кол-во слов в предложении.
        max_position: Макс. допустимая позиция.
        seed: Seed для воспроизводимости.
    
    Returns:
        DataFrame с колонками text, label.
    """
    random.seed(seed)
    samples = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating samples"):
        text = str(row["full_text"]).strip()
        words = text.split()
        if len(words) < min_words:
            continue
        positions = list(range(min(len(words), max_position)))
        positions = [p for p in positions if p < len(words)]
        if not positions:
            continue
        chosen = random.sample(positions, min(n_per_sentence, len(positions)))
        chosen = list(set(chosen))
        for pos in chosen:
            if pos >= max_position:
                continue
            new_words = words[:pos] + words[pos + 1:]
            new_text = " ".join(new_words)
            samples.append({"text": new_text, "label": pos})
    return pd.DataFrame(samples)


# Если train_df нет (ячейку EDA пропустили или она упала) — загружаем/создаём данные
if "train_df" not in globals():
    print("⚠️ train_df не найден. Загружаю данные...")
    if os.path.exists(CFG["train_path"]) and os.path.exists(CFG["test_path"]):
        train_df = pd.read_csv(CFG["train_path"])
        test_df = pd.read_csv(CFG["test_path"])
    else:
        print("   Файлы не найдены. Использую синтетические данные.")
        _s = ["Мен дүкенге барып келдім", "Сәлем қалайсың", "Бүгін жақсы күн", "Ол мектепке барды", "Біз кітап оқыдық"] * 200
        train_df = pd.DataFrame({"full_text": _s})
        test_df = pd.DataFrame({"ID": range(100), "text": ["Мен барып келдім", "Сәлем қалайсың"] * 50})
    train_df["n_words"] = train_df["full_text"].str.split().str.len()
    test_df["n_words"] = test_df["text"].str.split().str.len()
    CFG["max_position"] = int(test_df["n_words"].max()) + 2
    print(f"   train_df: {len(train_df)} строк, test_df: {len(test_df)} строк, max_position: {CFG['max_position']}\n")

t0 = time.time()
train_samples = generate_training_samples(
    train_df,
    n_per_sentence=CFG["n_samples_per_sentence"],
    min_words=CFG["min_words"],
    max_position=CFG["max_position"],
    seed=CFG["seed"],
)
print(f"Время генерации: {time.time() - t0:.1f} сек")
print(f"Размер: {len(train_samples)}")
print("\nРаспределение по лейблам (top 10):")
print(train_samples["label"].value_counts().head(10))
print("\nПримеры (sample 5):")
print(train_samples.sample(5, random_state=CFG["seed"]).to_string())

## Train / Val split

Разбиваем сгенерированные данные на train и validation.

In [ ]:
train_split, val_split = train_test_split(
    train_samples,
    test_size=CFG["val_size"],
    random_state=CFG["seed"],
    shuffle=True,
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
print(f"Train split: {len(train_split)}")
print(f"Val split: {len(val_split)}")

## Dataset и токенизатор

**xlm-roberta-base** обучен на 100 языках, включая казахский (CommonCrawl — живой интернет, не только Wikipedia). Используем его для токенизации и классификации позиции.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])


class KazakhPositionDataset(Dataset):
    """
    Dataset для предсказания позиции пропущенного слова в казахском предложении.
    """

    def __init__(self, texts: list, labels: list = None, max_len: int = 128):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors=None,
        )
        out = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        if self.labels is not None:
            out["labels"] = int(self.labels[idx])
        return out


train_dataset = KazakhPositionDataset(
    train_split["text"].tolist(),
    train_split["label"].tolist(),
    max_len=CFG["max_len"],
)
val_dataset = KazakhPositionDataset(
    val_split["text"].tolist(),
    val_split["label"].tolist(),
    max_len=CFG["max_len"],
)
test_dataset = KazakhPositionDataset(
    test_df["text"].tolist(),
    labels=None,
    max_len=CFG["max_len"],
)

print("Пример элемента train_dataset[0]:")
print(train_dataset[0])

## Модель

Используем **Sequence Classification**: num_labels = max_position. Позиции не имеют линейной зависимости с точки зрения синтаксиса — классификация предпочтительнее регрессии.

In [ ]:
# Модель создаётся внутри цикла обучения (для ансамбля — несколько моделей)
# Выводим информацию о размере модели
_dummy = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"],
    num_labels=CFG["max_position"],
    ignore_mismatched_sizes=True,
)
total = sum(p.numel() for p in _dummy.parameters())
print(f"Total params: {total:,}")
del _dummy
torch.cuda.empty_cache()

## Обучение

Trainer с compute_metrics, DataCollatorWithPadding и настроенными TrainingArguments.

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}


data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

# Обучение ансамбля: для каждого seed — своя модель
output_base = os.path.dirname(CFG["output_dir"])
for idx, seed in enumerate(CFG["ensemble_seeds"]):
    set_seed(seed)
    torch.manual_seed(seed)
    out_dir = os.path.join(output_base, f"model_seed{seed}")
    os.makedirs(out_dir, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        CFG["model_name"],
        num_labels=CFG["max_position"],
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=CFG["batch_size"],
        per_device_eval_batch_size=CFG["batch_size"] * 2,
        num_train_epochs=CFG["epochs"],
        learning_rate=CFG["lr"],
        weight_decay=CFG["weight_decay"],
        warmup_ratio=CFG["warmup_ratio"],
        fp16=CFG["fp16"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        logging_steps=500,
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    t0 = time.time()
    trainer.train()
    trainer.save_model(out_dir)
    print(f"Model seed {seed}: {(time.time() - t0) / 60:.1f} мин")

    if idx < len(CFG["ensemble_seeds"]) - 1:
        del model, trainer
        torch.cuda.empty_cache()

## Результаты обучения

Оценка на val, построение графиков loss и accuracy.

In [ ]:
try:
    metrics = trainer.evaluate()
    print("Validation metrics:", metrics)
except RuntimeError as e:
    if "on_train_begin" in str(e):
        history = trainer.state.log_history
        last_eval = [h for h in reversed(history) if "eval_accuracy" in h]
        metrics = last_eval[0] if last_eval else {}
        print("Validation metrics (from log_history):", metrics)
    else:
        raise

history = trainer.state.log_history
train_loss = [h["loss"] for h in history if "loss" in h]
val_acc = [h["eval_accuracy"] for h in history if "eval_accuracy" in h]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_loss, color="blue")
axes[0].set_title("Train loss по шагам")
axes[0].set_xlabel("Step")
axes[1].plot(val_acc, color="green", marker="o")
axes[1].set_title("Val accuracy по эпохам")
axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.show()

torch.cuda.empty_cache()

## Инференс

Предсказание позиций для test. Обязателен clamping: pred не должна превышать кол-во слов в оригинале.

In [ ]:
def predict_positions(
    model,
    tokenizer,
    texts: list,
    original_texts: list,
    batch_size: int = 64,
    device: str = "cuda",
) -> list:
    """
    Предсказывает позицию пропущенного слова для каждого текста.

    Args:
        model: Обученная модель.
        tokenizer: Токенизатор.
        texts: Список текстов с пропуском.
        original_texts: Оригинальные тексты (с пропуском) для clamping.
        batch_size: Размер батча.
        device: Устройство.

    Returns:
        Список int — предсказанные позиции.
    """
    model.eval()
    model.to(device)
    predictions = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Inference"):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            truncation=True,
            max_length=CFG["max_len"],
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        preds = out.logits.argmax(dim=-1).cpu().tolist()
        orig_batch = original_texts[i : i + batch_size]
        for j, pred in enumerate(preds):
            n_words = len(orig_batch[j].split())
            clamped = min(int(pred), n_words)
            predictions.append(clamped)
    return predictions


def get_logits(model, tokenizer, texts, batch_size=64, device="cuda") -> np.ndarray:
    model.eval()
    model.to(device)
    all_logits = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Logits"):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        all_logits.append(out.logits.cpu().numpy())
    return np.vstack(all_logits)


t0 = time.time()
output_base = os.path.dirname(CFG["output_dir"])
use_ens = CFG.get("use_ensemble", False) and len(CFG.get("ensemble_seeds", [42])) >= 2
texts_list = test_df["text"].tolist()

if use_ens:
    all_logits = []
    for seed in CFG["ensemble_seeds"]:
        model_dir = os.path.join(output_base, f"model_seed{seed}")
        m = AutoModelForSequenceClassification.from_pretrained(model_dir, num_labels=CFG["max_position"])
        all_logits.append(get_logits(m, tokenizer, texts_list, 64, "cuda"))
        del m
        torch.cuda.empty_cache()
    ensemble_logits = np.mean(all_logits, axis=0)
    raw_preds = np.argmax(ensemble_logits, axis=-1)
    predictions = [min(int(p), len(t.split())) for p, t in zip(raw_preds, texts_list)]
    submission_path = CFG["ensemble_submission_path"]
    print("Ансамбль: 2 модели")
else:
    predictions = predict_positions(model, tokenizer, texts_list, texts_list, batch_size=64, device="cuda")
    submission_path = CFG["submission_path"]

print(f"Время инференса: {time.time() - t0:.1f} сек")
print("Распределение предсказаний (top 15):")
print(pd.Series(predictions).value_counts().sort_index().head(15))

## Submission

Формируем сабмит и сохраняем в CSV.

In [ ]:
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "missing_word_position": predictions,
})

assert len(submission) == len(test_df)
assert submission["missing_word_position"].min() >= 0
assert pd.api.types.is_integer_dtype(submission["missing_word_position"]), "missing_word_position должен быть integer"

save_path = CFG["ensemble_submission_path"] if (CFG.get("use_ensemble") and len(CFG.get("ensemble_seeds", [])) >= 2) else CFG["submission_path"]
submission.to_csv(save_path, index=False)
print(f"Saved: {save_path}")
print(submission.head(10).to_string(index=False))

## Ансамбль

Ансамбль 2 моделей (seeds 42 и 123) уже включён: при `use_ensemble=True` инференс усредняет logits и сохраняет в `submission_ensemble.csv`.

In [ ]:
# Ансамбль выполняется в ячейке «Инференс» (get_logits + усреднение).
# Сабмит ансамбля сохраняется в CFG["ensemble_submission_path"].
print("Ансамбль:", "включён" if CFG["use_ensemble"] else "выключен")

## Итог

| Метрика | Значение |
|---------|----------|
| Val Accuracy | см. trainer.evaluate() |

### Советы по улучшению

- Увеличить `n_samples_per_sentence` до 5
- Ансамбль с другой моделью или другим seed
- `xlm-roberta-large` на Yandex Cloud GPU
- Span insertion scoring через MLM perplexity